##### Use https://docs.ragas.io/en/stable/concepts/ to estimate rag response quality

In [1]:
import openai

from langsmith import Client
from qdrant_client import QdrantClient

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper

/Users/apple/Downloads/Maven - End-to-End AI Engineering Bootcamp 2026-2/my-codes/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/apple/Downloads/Maven - End-to-End AI Engineering Bootcamp 2026-2/my-codes/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/apple/Downloads/Maven - End-to-End AI Engineering Bootcamp 2026-2/my-codes/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end 

#### Download an example reference data point from LangSmith

In [2]:
import os


client = Client(
        api_key=os.environ["LANGSMITH_API_KEY"]
)

Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your internet connection. ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /info (Caused by ConnectTimeoutError(<HTTPSConnection(host=\'api.smith.langchain.com\', port=443) at 0x1243f31c0>, \'Connection to api.smith.langchain.com timed out. (connect timeout=10.0)\'))"))\nContent-Length: None\nAPI Key: lsv2_********************************************5f')


In [5]:
dataset = client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

LangSmithConnectionError: Connection error caused failure to GET /datasets in LangSmith API. Please confirm your internet connection. ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host='api.smith.langchain.com', port=443): Max retries exceeded with url: /datasets?limit=1&name=rag-evaluation-dataset (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.smith.langchain.com', port=443) at 0x125fbb370>, 'Connection to api.smith.langchain.com timed out. (connect timeout=10.0)'))"))
Content-Length: None
API Key: lsv2_********************************************5f

In [ ]:
dataset

Dataset(name='rag-evaluation-dataset', description='Dataset for evaluating RAG pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('7e7ee270-8148-4f79-860e-f994e6584230'), created_at=datetime.datetime(2026, 5, 12, 20, 58, 27, 323579, tzinfo=datetime.timezone.utc), modified_at=datetime.datetime(2026, 5, 12, 20, 58, 27, 323579, tzinfo=datetime.timezone.utc), example_count=34, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-13.7.8-x86_64-i386-64bit', 'sdk_version': '0.4.37', 'runtime_version': '3.9.6', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [ ]:
list(client.list_examples(dataset_id=dataset.id, limit = 10))[0].outputs

{'ground_truth': 'Mr. Misunderstood On The Rocks: Live And Mostly Unplugged',
 'reference_context_ids': ['B01M9IAW2Z'],
 'reference_description': ['187 by Samuel L. Jackson ']}

In [ ]:
list(client.list_examples(dataset_id=dataset.id, limit = 10))[0].inputs

{'question': "Name the item that includes 'Mr. Misunderstood On The Rocks' live release."}

In [ ]:
reference_inputs = list(client.list_examples(dataset_id=dataset.id, limit = 10))[0].inputs
reference_outputs = list(client.list_examples(dataset_id=dataset.id, limit = 10))[0].outputs

#### RAG Pipeline

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding

def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name = "Amazon-items-collection-00",
        query = query_embedding,
        limit = k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):
    formatted_context = ""

    for chunk_id, chunk, rating in zip(
         context["retrieved_context_ids"], 
         context["retrieved_context"],  
         context["retrieved_context_ratings"],
    ):
        formatted_context += f"- ID: {chunk_id}, rating: {rating}, description: {chunk}\n" 

    return formatted_context

def build_prompt(preprocessed_context, question):
    prompt = f"""
        You are a shopping assistant that can answer questions about the products in stock.

        You will be given a question and list of context

        Instructions:
        - You need to answer question based on the provided context only.
        - Never use word context and refer to it as the available products.

        Context:
        {preprocessed_context}

        Question:
        {question}
    """

    return prompt

def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "system", "content": prompt}],
        reasoning_effort="minimal"
    )
    
    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_result={
        "answer": answer,
        "question": question,
        "retrieved_context_ids":retrieved_context["retrieved_context_ids"],
        "retrieved_context":retrieved_context["retrieved_context"],
        "similarity_scores":retrieved_context["similarity_scores"],
    }

    return final_result




In [ ]:
rag_pipeline("What is the best music that I can hear?", top_k=5)

##### RAGAS metrics

In [ ]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevency

In [ ]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

In [ ]:
reference_inputs

In [ ]:
reference_outputs

In [ ]:
result = rag_pipeline(reference_inputs["question"])
result

In [ ]:
async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input = run["question"],
        response = run["answer"],
        retrieved_context = run["retrieved_context"]
    )

    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_faithfulness(result, "")

In [ ]:
async def ragas_response_relevancy(run, example):
    sample = SingleTurnSample(
        user_input = run["question"],
        response = run["answer"],
        retrieved_context = run["retrieved_context"] 
    )

    scorer = ResponseRelevency(llm= ragas_llm, embeddings = ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_response_relevancy(result, "")

In [ ]:
async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids = run["retrieved_context_ids"],
        reference_context_ids = example["reference_context_ids"],
    )
    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_precision_id_based(result, reference_outputs)

In [ ]:
async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids = run["retrieved_context_ids"],
        reference_context_ids = example["reference_context_ids"],
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [ ]:
await ragas_context_recall_id_based(result, reference_outputs)